# 01 — Bread Basket EDA & Preprocess (Colab)

카페 추천 시스템(Item-based CF + Item2Vec) 학습을 위한 외부 데이터 전처리 노트북.

## 데이터 업로드 방법 (택 1)

**방법 A — 직접 업로드 (가장 빠름)**
- 좌측 사이드바의 **파일 (📁) → 업로드 (⬆)** 아이콘 클릭.
- `bread basket.csv` 파일을 그대로 업로드.
- 업로드 위치: `/content/bread basket.csv` (Colab 기본 작업 디렉토리).
- 파일은 **세션이 끝나면 사라짐** — 매번 다시 올려야 함.

**방법 B — Google Drive 마운트 (반복 사용 시 권장)**
1. Google Drive에 `Adaptive_Kiosk/create_data/` 폴더 만들고 그 안에 `bread basket.csv` 업로드.
2. 아래 "0. Imports & Paths" 셀의 `USE_GDRIVE = True`로 바꾸고 실행.
3. 인증 팝업 → 허용 → Drive 마운트.

**방법 C — 외부 URL 직접 다운로드**
- Kaggle API key를 Colab에 등록하면 `!kaggle datasets download` 가능.
- 본 노트북에서는 다루지 않음 (방법 A/B로 충분).

## 출력

- `output/transactions_clean.csv`
- `output/menus_meta.csv`

출력 경로:
- 방법 A → `/content/output/`
- 방법 B → `/content/drive/MyDrive/Adaptive_Kiosk/create_data/output/`

노트북 실행 후 위 두 파일을 다운로드 받아 로컬의 `Adaptive_Kiosk/create_data/output/`에 두면 다음 단계에서 사용 가능.

## 0. Imports & Paths

In [ ]:
from pathlib import Path
from collections import Counter
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

# ====================================================================
# 데이터 위치 선택. 위쪽 markdown 설명 참고.
#   USE_GDRIVE = False  → 직접 업로드 모드 (/content/bread basket.csv)
#   USE_GDRIVE = True   → Google Drive 마운트 모드
# ====================================================================
USE_GDRIVE = False

if USE_GDRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = Path("/content/drive/MyDrive/Adaptive_Kiosk/create_data")
else:
    DATA_DIR = Path("/content")

RAW_PATH = DATA_DIR / "bread basket.csv"
OUTPUT_DIR = DATA_DIR / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("DATA_DIR :", DATA_DIR)
print("RAW_PATH :", RAW_PATH, "exists?", RAW_PATH.exists())
print("OUTPUT_DIR:", OUTPUT_DIR)

if not RAW_PATH.exists():
    print("\n⚠️  파일이 없습니다. 위 markdown의 'A. 직접 업로드' 또는 'B. Drive 마운트' 단계를 따라 'bread basket.csv'를 위 RAW_PATH 위치에 두세요.")

## 1. Load

In [ ]:
raw_df = pd.read_csv(RAW_PATH)
print("shape:", raw_df.shape)
print("columns:", list(raw_df.columns))
raw_df.head(10)

In [ ]:
raw_df.info()

## 2. 결측치 / 중복 / 가비지 처리

- `Item`이 `"NONE"` 같은 가비지인 row 제거.
- 같은 `(Transaction, Item)`이 여러 행으로 등장 → 한 영수증 안에서 같은 메뉴 N개 산 의미. 추후 `quantity`로 합산.
- 시간 결측 row가 있으면 제거 (없을 가능성 높음).

In [ ]:
df = raw_df.copy()

df.columns = [c.strip() for c in df.columns]
df["Item"] = df["Item"].astype(str).str.strip()

before = len(df)
df = df[~df["Item"].str.upper().isin({"NONE", ""})].copy()
df = df.dropna(subset=["Transaction", "Item", "date_time"]).copy()
after = len(df)

print(f"행 수 변화: {before:,} → {after:,}  (제거 {before-after:,})")

In [ ]:
agg = (
    df.groupby(["Transaction", "Item", "date_time", "period_day", "weekday_weekend"])
    .size()
    .reset_index(name="quantity")
)
print("집계 후 shape:", agg.shape)
agg.head()

## 3. 시간 컬럼 정제

In [ ]:
agg["datetime"] = pd.to_datetime(agg["date_time"], errors="coerce", dayfirst=True)
n_bad = agg["datetime"].isna().sum()
print("datetime 파싱 실패:", n_bad)
agg = agg.dropna(subset=["datetime"]).copy()

agg["hour"] = agg["datetime"].dt.hour
agg["day_of_week"] = agg["datetime"].dt.day_name()
agg["date"] = agg["datetime"].dt.date

print("기간:", agg["datetime"].min(), "~", agg["datetime"].max())
agg.head()

## 4. EDA

### 4-1. 기본 통계

In [ ]:
n_orders = agg["Transaction"].nunique()
n_items = agg["Item"].nunique()
n_lines = len(agg)

print(f"distinct orders : {n_orders:,}")
print(f"distinct items  : {n_items:,}")
print(f"total lines     : {n_lines:,}")

### 4-2. 다중 라인 분포 — Item2Vec 학습 가능성 핵심 지표

`>=2 distinct items per order` 비율이 30% 이상이어야 학습 의미 있음.

In [ ]:
lines_per_order = agg.groupby("Transaction")["Item"].nunique()
print(lines_per_order.describe())
print(f"\norders with >=2 distinct items: "
      f"{int((lines_per_order>=2).sum()):,} ({(lines_per_order>=2).mean()*100:.1f}%)")
print(f"avg distinct items per order : {lines_per_order.mean():.2f}")
print(f"max distinct items per order : {lines_per_order.max()}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
lines_per_order.value_counts().sort_index().plot(kind="bar", ax=ax, color="#0f172a")
ax.set_xlabel("distinct items per order")
ax.set_ylabel("# orders")
ax.set_title("Distribution: distinct items per order")
plt.tight_layout()
plt.show()

### 4-3. 시간대별 / 요일별 주문량

In [ ]:
orders_by_hour = (
    agg.groupby("Transaction")["hour"].first().value_counts().sort_index()
)
fig, ax = plt.subplots(figsize=(10, 4))
orders_by_hour.plot(kind="bar", ax=ax, color="#f59e0b")
ax.set_xlabel("hour of day")
ax.set_ylabel("# orders")
ax.set_title("Orders by hour of day")
plt.tight_layout()
plt.show()

In [ ]:
dow_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
orders_by_dow = (
    agg.groupby("Transaction")["day_of_week"].first().value_counts().reindex(dow_order)
)
fig, ax = plt.subplots(figsize=(8, 4))
orders_by_dow.plot(kind="bar", ax=ax, color="#34d399")
ax.set_xlabel("day of week")
ax.set_ylabel("# orders")
ax.set_title("Orders by day of week")
plt.tight_layout()
plt.show()

In [ ]:
orders_by_period = (
    agg.groupby("Transaction")["period_day"].first().value_counts()
)
orders_by_weekend = (
    agg.groupby("Transaction")["weekday_weekend"].first().value_counts()
)
print("period_day 분포:")
print(orders_by_period)
print("\nweekday/weekend 분포:")
print(orders_by_weekend)

### 4-4. 메뉴 인기 TOP-N

In [ ]:
item_popularity = agg.groupby("Item")["quantity"].sum().sort_values(ascending=False)
print("distinct items:", len(item_popularity))
top_n = 25
top_items = item_popularity.head(top_n)

fig, ax = plt.subplots(figsize=(8, 8))
top_items[::-1].plot(kind="barh", ax=ax, color="#0f172a")
ax.set_xlabel("sum of quantity")
ax.set_title(f"Top {top_n} items")
plt.tight_layout()
plt.show()

top_items

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(range(len(item_popularity)), item_popularity.values, color="#475569")
ax.set_xlabel("item rank")
ax.set_ylabel("sum of quantity (log)")
ax.set_title("Item popularity (long tail)")
ax.set_yscale("log")
plt.tight_layout()
plt.show()

### 4-5. 동시구매 페어 TOP-N

In [ ]:
pair_counter = Counter()
grouped = agg.groupby("Transaction")["Item"].apply(lambda s: sorted(set(s)))
for items in grouped:
    if len(items) < 2:
        continue
    for a, b in combinations(items, 2):
        pair_counter[(a, b)] += 1

pairs_df = (
    pd.DataFrame(
        [(a, b, c) for (a, b), c in pair_counter.items()],
        columns=["item_a", "item_b", "co_count"],
    )
    .sort_values("co_count", ascending=False)
    .reset_index(drop=True)
)
print(f"distinct co-purchase pairs: {len(pairs_df):,}")
pairs_df.head(20)

## 5. 메뉴 메타 작성

Bread Basket에는 카테고리/카페인/온도 정보가 없다. 메뉴 이름 기반 휴리스틱으로 1차 라벨링.
이 라벨은 추후 hybrid (CF + content) 단계에서 활용 예정.

**주의**: 1차 매핑은 키워드 기반으로 거칠게 한다. 사용자가 결과를 보고 보강하기로 한다.

In [ ]:
CATEGORY_KEYWORDS = [
    ("coffee",     ["coffee", "espresso", "latte", "mocha"]),
    ("tea",        ["tea", "chai"]),
    ("hot_drink",  ["hot chocolate", "chocolates"]),
    ("juice",      ["juice"]),
    ("smoothie",   ["smoothie", "frappe"]),
    ("bread",      ["bread", "toast", "focaccia", "baguette"]),
    ("pastry",     ["pastry", "croissant", "medialuna", "scone", "muffin", "brownie", "cake", "tartine", "alfajores", "cookies", "truffles", "jammie dodgers"]),
    ("sandwich",   ["sandwich", "toastie"]),
    ("soup",       ["soup"]),
    ("salad",      ["salad"]),
    ("breakfast",  ["breakfast", "granola"]),
    ("savory",     ["farm house", "frittata"]),
]

CAFFEINATED_HINTS = {"coffee", "espresso", "latte", "mocha", "chai", "tea"}
COLD_HINTS = {"juice", "smoothie", "frappe", "iced"}
HOT_HINTS = {"hot", "latte", "mocha", "espresso"}


def classify(name: str):
    n = name.lower()
    category = "other"
    for cat, keywords in CATEGORY_KEYWORDS:
        if any(k in n for k in keywords):
            category = cat
            break
    is_caffeinated = any(k in n for k in CAFFEINATED_HINTS)
    if any(k in n for k in COLD_HINTS):
        temp = "cold"
    elif any(k in n for k in HOT_HINTS):
        temp = "hot"
    else:
        temp = None
    return category, is_caffeinated, temp


menus_meta = (
    pd.DataFrame({"Item": item_popularity.index, "total_quantity": item_popularity.values})
    .reset_index(drop=True)
)
menus_meta[["category", "is_caffeinated", "serving_temperature"]] = menus_meta["Item"].apply(
    lambda x: pd.Series(classify(x))
)
menus_meta.head(20)

In [ ]:
menus_meta["category"].value_counts()

In [ ]:
menus_meta[menus_meta["category"] == "other"].sort_values("total_quantity", ascending=False)

## 6. 저장 + Colab에서 다운로드

- `transactions_clean.csv`, `menus_meta.csv` 두 파일을 `OUTPUT_DIR`에 저장.
- 직접 업로드 모드(`USE_GDRIVE=False`)에서는 **마지막 셀의 `files.download(...)` 으로 로컬에 내려받음**.
- Drive 모드(`USE_GDRIVE=True`)에서는 Drive에 그대로 보존되어 다음 노트북에서 재로드 가능.

In [ ]:
transactions_clean = agg[
    ["Transaction", "Item", "quantity", "datetime", "hour", "day_of_week", "period_day", "weekday_weekend"]
].copy()

tx_path = OUTPUT_DIR / "transactions_clean.csv"
menu_path = OUTPUT_DIR / "menus_meta.csv"

transactions_clean.to_csv(tx_path, index=False, encoding="utf-8")
menus_meta.to_csv(menu_path, index=False, encoding="utf-8")

print("saved:", tx_path)
print("saved:", menu_path)

In [ ]:
# 직접 업로드 모드일 때 로컬로 다운로드. Drive 모드에서는 스킵해도 됨.
if not USE_GDRIVE:
    try:
        from google.colab import files
        files.download(str(tx_path))
        files.download(str(menu_path))
    except Exception as e:
        print("다운로드 스킵 (Colab 외 환경):", e)

## 7. 다음 단계

1. 위 EDA 결과(다중 라인 비율, 시간대 분포, TOP-N, 동시구매 페어, 카테고리 분포, `other` 후보)를 사용자가 확인.
2. 보강이 필요한 부분(예: 카테고리 키워드 추가, 가비지 메뉴 제거 등)을 본 노트북 안에서 반영.
3. 만족스러우면 다음 노트북(`02_train_models.ipynb`)에서 Item-based CF + Item2Vec 학습 진행.

사용자 의도: **"전처리 및 통계 확인 먼저 → 결과 보고 → 적합하게 전처리 변경 → 모델"**